# v3 HRL vs. Flat MARL Analysis

This notebook loads the tournament results produced by `training/evaluate_hrl.py`
and generates publication-ready plots to answer the primary v3 research question:

> **Does the three-echelon HRL architecture outperform the v2 flat MAPPO baseline
> in a controlled 4v4 tournament?**

## Evaluation Protocol

| Parameter | Value |
|-----------|-------|
| Scenario | 4v4 (2 Blue brigades × 2 battalions each) |
| Episodes per arm | ≥ 100 |
| Confidence intervals | Bootstrapped 95 % CI (2 000 resamples) |
| Red policy | Stationary (default) or random |
| Outcome criterion | `info["winner"] == "blue"` for HRL; agent-survival for Flat MARL |

## Usage

1. Run the tournament:
   ```bash
   python training/evaluate_hrl.py \\
       --division-checkpoint checkpoints/division/ppo_division_final.zip \\
       --mappo-checkpoint    checkpoints/mappo/team_snapshot_v000100.pt \\
       --n-episodes 100 --seed 0 \\
       --output results/hrl_vs_marl.json
   ```
2. Set `RESULTS_PATH` below to the output file path.
3. Run all cells top-to-bottom.

If no saved results file is available the *Synthetic data* cell generates
placeholder results so all plots render correctly.

In [ ]:
# ── Dependencies ─────────────────────────────────────────────────────────────
import sys
from pathlib import Path

# Add project root to sys.path so local imports work from notebooks/ dir
PROJECT_ROOT = Path(".").resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Path to the tournament results JSON (produced by training/evaluate_hrl.py)
RESULTS_PATH = PROJECT_ROOT / "results" / "hrl_vs_marl.json"

print(f"Project root : {PROJECT_ROOT}")
print(f"Results path : {RESULTS_PATH}")
print(f"Results exist: {RESULTS_PATH.exists()}")

## 1 — Load Results

Load the tournament JSON, falling back to synthetic data when the file is absent.

In [ ]:
if RESULTS_PATH.exists():
    with open(RESULTS_PATH) as f:
        summary = json.load(f)
    print("✅ Loaded results from", RESULTS_PATH)
else:
    # ── Synthetic placeholder ───────────────────────────────────────────────
    # Replace with real results when a training run is available.
    print("⚠️  Results file not found — using synthetic placeholder data.")
    from training.evaluate_hrl import bootstrap_ci, TournamentResult, run_tournament

    _rng = np.random.default_rng(42)

    # Simulate HRL: slightly higher win rate
    hrl_outcomes = [1] * 55 + [0] * 15 + [-1] * 30
    _rng.shuffle(hrl_outcomes)
    hrl_ci = bootstrap_ci(hrl_outcomes, n_bootstrap=2000, rng=np.random.default_rng(0))
    hrl = TournamentResult(
        wins=55, draws=15, losses=30, n_episodes=100,
        win_rate=0.55, draw_rate=0.15, loss_rate=0.30,
        ci_lower=hrl_ci[0], ci_upper=hrl_ci[1],
        ci_confidence=0.95, label="HRL",
    )

    # Simulate Flat MARL
    flat_outcomes = [1] * 42 + [0] * 18 + [-1] * 40
    _rng.shuffle(flat_outcomes)
    flat_ci = bootstrap_ci(flat_outcomes, n_bootstrap=2000, rng=np.random.default_rng(1))
    flat = TournamentResult(
        wins=42, draws=18, losses=40, n_episodes=100,
        win_rate=0.42, draw_rate=0.18, loss_rate=0.40,
        ci_lower=flat_ci[0], ci_upper=flat_ci[1],
        ci_confidence=0.95, label="FlatMARL",
    )

    summary = run_tournament(hrl, flat)

hrl_data  = summary["hrl"]
flat_data = summary["flat_marl"]

print(f"\nHRL  — win_rate={hrl_data['win_rate']:.3f}  "
      f"CI=[{hrl_data['ci_lower']:.3f}, {hrl_data['ci_upper']:.3f}]")
print(f"Flat — win_rate={flat_data['win_rate']:.3f}  "
      f"CI=[{flat_data['ci_lower']:.3f}, {flat_data['ci_upper']:.3f}]")
print(f"Δ win_rate = {summary['delta_win_rate']:+.4f}")
print(f"Conclusion : {summary['conclusion']}")

## 2 — Win Rate Comparison with Bootstrapped CIs

In [ ]:
labels   = [hrl_data["label"],  flat_data["label"]]
win_rates = [hrl_data["win_rate"], flat_data["win_rate"]]
ci_lowers = [hrl_data["ci_lower"], flat_data["ci_lower"]]
ci_uppers = [hrl_data["ci_upper"], flat_data["ci_upper"]]
colors    = ["steelblue", "darkorange"]

yerr_lo = [wr - lo for wr, lo in zip(win_rates, ci_lowers)]
yerr_hi = [hi - wr for wr, hi in zip(win_rates, ci_uppers)]

fig, ax = plt.subplots(figsize=(6, 5))
bars = ax.bar(
    labels, win_rates,
    color=colors, alpha=0.8, edgecolor="white", width=0.4,
    yerr=[yerr_lo, yerr_hi], capsize=8, error_kw={"elinewidth": 2, "ecolor": "black"},
)
ax.set_ylim(0, 1.05)
ax.set_ylabel("Blue Win Rate", fontsize=12)
ax.set_title(
    f"HRL vs. Flat MARL — 4v4 Tournament\n"
    f"({hrl_data['n_episodes']} episodes per arm, "
    f"{int(hrl_data['ci_confidence'] * 100):.0f}% bootstrapped CI)",
    fontsize=11,
)
for bar, wr in zip(bars, win_rates):
    ax.text(
        bar.get_x() + bar.get_width() / 2.0,
        wr + max(yerr_hi) + 0.02,
        f"{wr:.2f}",
        ha="center", va="bottom", fontsize=11, fontweight="bold",
    )
ax.grid(axis="y", alpha=0.3)
ax.spines[["top", "right"]].set_visible(False)

# Annotate conclusion
ax.text(
    0.5, 0.03, summary["conclusion"],
    transform=ax.transAxes, ha="center", va="bottom",
    fontsize=9, color="dimgray", style="italic",
)

plt.tight_layout()
plt.savefig("hrl_vs_marl_winrate.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Saved hrl_vs_marl_winrate.png")

## 3 — Outcome Distribution Breakdown

In [ ]:
x = np.arange(2)
width = 0.25

wins_vals  = [hrl_data["win_rate"],  flat_data["win_rate"]]
draw_vals  = [hrl_data["draw_rate"], flat_data["draw_rate"]]
loss_vals  = [hrl_data["loss_rate"], flat_data["loss_rate"]]

fig, ax = plt.subplots(figsize=(7, 5))
b1 = ax.bar(x - width, wins_vals, width, label="Win",  color="seagreen",  alpha=0.85)
b2 = ax.bar(x,         draw_vals, width, label="Draw", color="steelblue", alpha=0.85)
b3 = ax.bar(x + width, loss_vals, width, label="Loss", color="firebrick", alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels([hrl_data["label"], flat_data["label"]], fontsize=12)
ax.set_ylim(0, 1.1)
ax.set_ylabel("Rate", fontsize=12)
ax.set_title("Outcome Distribution — HRL vs. Flat MARL", fontsize=12)
ax.legend(fontsize=10)
ax.grid(axis="y", alpha=0.3)
ax.spines[["top", "right"]].set_visible(False)

for bars in [b1, b2, b3]:
    for bar in bars:
        h = bar.get_height()
        if h > 0.02:
            ax.text(
                bar.get_x() + bar.get_width() / 2.0, h + 0.01,
                f"{h:.2f}", ha="center", va="bottom", fontsize=8,
            )

plt.tight_layout()
plt.savefig("hrl_vs_marl_outcomes.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Saved hrl_vs_marl_outcomes.png")

## 4 — Bootstrapped CI Distribution

Visualise the full bootstrap distribution for each arm to show uncertainty.

In [ ]:
from training.evaluate_hrl import bootstrap_ci

def _synth_outcomes(wins, draws, losses):
    """Reconstruct outcome list from aggregate counts."""
    return ([1] * wins) + ([0] * draws) + ([-1] * losses)

hrl_outcomes  = _synth_outcomes(hrl_data["wins"],  hrl_data["draws"],  hrl_data["losses"])
flat_outcomes = _synth_outcomes(flat_data["wins"], flat_data["draws"], flat_data["losses"])

N_BOOT = 2000
rng = np.random.default_rng(0)

def _boot_distribution(outcomes, n_bootstrap=N_BOOT):
    arr = np.asarray(outcomes, dtype=np.float32)
    win_flags = (arr == 1).astype(np.float32)
    n = len(win_flags)
    rates = np.empty(n_bootstrap)
    for i in range(n_bootstrap):
        rates[i] = rng.choice(win_flags, size=n, replace=True).mean()
    return rates

hrl_boot  = _boot_distribution(hrl_outcomes)
flat_boot = _boot_distribution(flat_outcomes)

fig, ax = plt.subplots(figsize=(8, 4))
bins = np.linspace(0, 1, 41)

ax.hist(hrl_boot,  bins=bins, alpha=0.6, color="steelblue",  label=hrl_data["label"],  density=True)
ax.hist(flat_boot, bins=bins, alpha=0.6, color="darkorange", label=flat_data["label"], density=True)

ax.axvline(hrl_data["win_rate"],  color="steelblue",  linestyle="--", linewidth=2)
ax.axvline(flat_data["win_rate"], color="darkorange", linestyle="--", linewidth=2)

ax.set_xlabel("Bootstrapped Win Rate", fontsize=11)
ax.set_ylabel("Density", fontsize=11)
ax.set_title(
    f"Bootstrap Distribution of Win Rate ({N_BOOT:,} resamples)",
    fontsize=11,
)
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.savefig("hrl_vs_marl_bootstrap.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Saved hrl_vs_marl_bootstrap.png")

## 5 — Summary Statistics Table

In [ ]:
print("=" * 65)
print(f"{'Metric':<30} {'HRL':>15} {'Flat MARL':>15}")
print("-" * 65)
for key, fmt in [
    ("n_episodes",    "{:>15d}"),
    ("wins",          "{:>15d}"),
    ("draws",         "{:>15d}"),
    ("losses",        "{:>15d}"),
    ("win_rate",      "{:>15.3f}"),
    ("draw_rate",     "{:>15.3f}"),
    ("loss_rate",     "{:>15.3f}"),
    ("ci_lower",      "{:>15.3f}"),
    ("ci_upper",      "{:>15.3f}"),
    ("ci_confidence", "{:>15.2f}"),
]:
    hrl_val  = fmt.format(hrl_data[key])
    flat_val = fmt.format(flat_data[key])
    print(f"  {key:<28} {hrl_val} {flat_val}")
print("-" * 65)
print(f"  {'delta_win_rate':<28} {summary['delta_win_rate']:>15.4f}")
print(f"  {'ci_overlap':<28} {str(summary['ci_overlap']):>15}")
print("=" * 65)
print(f"\nConclusion: {summary['conclusion']}")

## 6 — (Optional) Run Fresh Tournament

Uncomment and adapt the cells below to run a fresh evaluation directly
from this notebook without using the CLI.

Requires:
* A division PPO checkpoint at `checkpoints/division/ppo_division_final.zip`
* A flat MARL MAPPO snapshot at `checkpoints/mappo/team_snapshot_v000100.pt`

In [ ]:
# from training.evaluate_hrl import run_hrl_episodes, run_flat_marl_episodes, run_tournament
#
# hrl_result = run_hrl_episodes(
#     n_episodes=100,
#     division_checkpoint=PROJECT_ROOT / "checkpoints/division/ppo_division_final.zip",
#     seed=0,
#     n_bootstrap=2000,
# )
#
# flat_result = run_flat_marl_episodes(
#     n_episodes=100,
#     mappo_checkpoint=PROJECT_ROOT / "checkpoints/mappo/team_snapshot_v000100.pt",
#     seed=0,
#     n_bootstrap=2000,
# )
#
# summary = run_tournament(hrl_result, flat_result)
# print(summary["conclusion"])